# Microsoft Q-Learning Experiment Lab — reproducible Python notebook

This notebook implements the same **tabular Q-learning agent** that runs on the workshop website through Pyodide. Rather than searching for a lucky validation return, it asks: **How do learning settings change Q-value evolution, the eventual policy, and stability?**

- Asset: Microsoft shares versus zero-yield cash; allocations **0%, 50%, 100%**.
- State: sign of 4-week and 12-week momentum plus previous allocation (**12 states, 36 Q-values**).
- Reward: weighted next-week stock return − proportional trading cost − downside penalty.
- Chronological **realized-return** split: 2013–2019 training, 2020–2022 validation, 2023–2024 final holdout. Momentum uses only information available when the decision is made.
- The same historical training weeks are repeated across passes. This does not generate independent samples.

The public historical price collection's adjustment quality has not been independently verified. This is for workshop instruction, **not investment research or advice**.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

root = next((p for p in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent)
             if (p/'asset_agent.py').exists()), None)
assert root is not None, "Run from repository root or notebooks/"
sys.path.insert(0,str(root))
from asset_agent import MicrosoftChallengeAgent, greedy

df=pd.read_csv(root/'data/msft_weekly_2013_2024.csv')
prices=df['close'].tolist()
dates=df['date'].tolist()
print(f"Data: {len(df)} weekly prices: {dates[0]} to {dates[-1]}")

## 1. Compare the same model under two learning rates

Change only **α**. Keep γ, ε, pass count, seed, transaction costs, reward and market data identical. Even if the final returns differ, that alone does not establish which learning rate is generally better.

In [ ]:
common=dict(gamma=0.90, epsilon=0.25, budget=100)
runs={}
for label,alpha in [('Small α',0.05),('Large α',0.40)]:
    agent=MicrosoftChallengeAgent(prices,dates,alpha=alpha,**common)
    agent.step_many(agent.budget*len(agent.train_t))
    results=agent.evaluate_segment('validation')
    runs[label]=(agent,results)
    rl=results[0]
    print(f"{label}: 2020–22 net return {rl['totalReturn']:.1%}, max drawdown {rl['maxDrawdown']:.1%}, allocation changes {rl['changes']}")
print("Train weeks:",len(runs['Small α'][0].train_t),"Validation weeks:",len(runs['Small α'][0].validation_t),"Final weeks:",len(runs['Small α'][0].test_t))

## 2. Watch the reward curves and Q-values

A training pass is a full replay of the original training history. The reward curve is the sum of the Q-learning reward over each pass—not the compounded portfolio return. Inspect a single state (state **6** = positive 4-week momentum, negative 12-week momentum, previously 0% invested) to see how its three action values evolve.

In [ ]:
fig,ax=plt.subplots(figsize=(10,4))
for label,(agent,_) in runs.items():
    ax.plot(np.arange(1,agent.passes+1),agent.greedy_history,label=label)
ax.set(title='Greedy policy reward on repeated training history',xlabel='Training pass',ylabel='Sum of weekly training rewards')
ax.legend();ax.grid(alpha=.2)
plt.show()

In [ ]:
state=6  # 4-week UP, 12-week DOWN, previous weight 0%
fig,ax=plt.subplots(figsize=(10,4))
for label,(agent,_) in runs.items():
    values=np.array([q[state] for q in agent.q_history])
    for a,weight in enumerate((0,50,100)):
        ax.plot(range(1,len(values)+1),values[:,a],label=f'{label}: {weight}% MSFT')
ax.set(title=f'Evolution of the three Q-values in state {state}',xlabel='Training pass',ylabel='Estimated Q-value')
ax.legend(ncol=2,fontsize=8);ax.grid(alpha=.2)
plt.show()

## 3. Compare what the agents actually learned

The policy selects the action with the largest Q-value in each state. An improved result on a single validation period is not necessarily a better *learning process*. Compare actions, turnover, and the annual decomposition.

In [ ]:
left,right=[runs[k][0] for k in runs]
states=[f"4w {'↑' if (i//3)&2 else '↓'}, 12w {'↑' if (i//3)&1 else '↓'}, prev {(i%3)*50}%" for i in range(12)]
policy=pd.DataFrame({
    'Market state':states,
    'Small α choice':[50*greedy(x) for x in left.q],
    'Large α choice':[50*greedy(x) for x in right.q]})
policy['Different?']=policy['Small α choice']!=policy['Large α choice']
print('Different preferred actions:',int(policy['Different?'].sum()),'of 12')
display(policy)

## 4. Check exploration-seed sensitivity

A single good run may depend on random exploratory actions. Evaluate three seeds without using the final holdout for tuning.

In [ ]:
seed_rows=[]
for seed in (17,29,43):
    agent=MicrosoftChallengeAgent(prices,dates,alpha=0.40,**common)
    agent.seed=seed
    agent.reset()
    agent.step_many(agent.budget*len(agent.train_t))
    r=agent.evaluate_segment('validation')[0]
    seed_rows.append(dict(seed=seed,return_pct=100*r['totalReturn'],drawdown_pct=100*r['maxDrawdown'],trades=r['changes']))
display(pd.DataFrame(seed_rows).round(2))

## 5. Final-test discipline

The 2023–2024 period was not used above. If this were an actual research project, write down a chosen policy and rationale *before* opening it. Avoid repeatedly making decisions based on the same final period. The website offers a single illustrative reveal per page session, but refreshing cannot enforce statistical independence.